In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv

# Mortgage Rate Enrichment

Enrich both the combined sold and listings datasets by merging in the national 30-year fixed mortgage rate from the St. Louis Federal Reserve (FRED). This introduces data analysts to fetching live economic data from a public API and performing a time-series join on a monthly key.

## Objective
Fetch the FRED MORTGAGE30US series, resample it from weekly to monthly frequency, and merge it onto both combined datasets using a year-month key derived from transaction dates.

## Background
The FRED MORTGAGE30US series is published weekly (every Thursday) by Freddie Mac via the St. Louis Federal Reserve. Because MLS transactions are analyzed by calendar month, interns must resample the weekly data to a monthly average before joining. The data can be fetched directly from FRED as a CSV—no API key required.

## Tasks

In [9]:
# Step 1 – Fetch the mortgage rate data from FRED
url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url, parse_dates=['observation_date'])
mortgage.columns = ['date', 'rate_30yr_fixed']

In [10]:
# Step 2 – Resample weekly rates to monthly averages
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = mortgage.groupby('year_month')['rate_30yr_fixed'].mean().reset_index()

In [11]:
# Step 3 – Create a matching year_month key on the MLS datasets
sold = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week1/sold.csv')
listings = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week1/listing.csv')

# Sold dataset — key off CloseDate
sold['year_month'] = pd.to_datetime(sold['CloseDate']).dt.to_period('M')

# Listings dataset — key off ListingContractDate
listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')

/var/folders/q8/sdky6zrj6y52rcdpgrsws4vm0000gn/T/ipykernel_99519/1037824410.py:2: DtypeWarning: Columns (0,1,9,64,78,79,80,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week1/sold.csv')
/var/folders/q8/sdky6zrj6y52rcdpgrsws4vm0000gn/T/ipykernel_99519/1037824410.py:3: DtypeWarning: Columns (2,43) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv('/Users/olee1232/Documents/IDXWork/Week1/listing.csv')


In [12]:
# Step 4 – Merge
sold_with_rates = sold.merge(mortgage_monthly, on='year_month', how='left')
listings_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

In [13]:
# Step 5 – Validate the merge
# Check for any unmatched rows (rate should not be null)
print(sold_with_rates['rate_30yr_fixed'].isnull().sum())
print(listings_with_rates['rate_30yr_fixed'].isnull().sum())

0
0


In [14]:
# Preview
print(sold_with_rates[['CloseDate', 'year_month', 'ClosePrice', 'rate_30yr_fixed']].head())

    CloseDate year_month  ClosePrice  rate_30yr_fixed
0  2024-01-26    2024-01    240000.0           6.6425
1  2024-01-05    2024-01    815000.0           6.6425
2  2024-01-05    2024-01    810000.0           6.6425
3  2024-01-30    2024-01    858000.0           6.6425
4  2024-01-29    2024-01   1890500.0           6.6425
